## Accessing Animal Tracking - Satellite Relay Tagging - Delayed Mode - Diagnostic (Parquet)
This Jupyter notebook demonstrates how to access and plot animal_diag_satellite_relay_tagging_location_qc_delayed_qc data, available as a [Parquet](https://parquet.apache.org) dataset stored on S3.

🔗 More information about the dataset is available [in the AODN metadata catalogue](https://catalogue-imos.aodn.org.au/geonetwork/srv/eng/catalog.search#/metadata/70f148b1-7040-4fad-944a-456413c95472).

📌 The source of truth for this notebook is maintained on [GitHub](https://github.com/aodn/aodn_cloud_optimised/tree/main/notebooks/animal_diag_satellite_relay_tagging_location_qc_delayed_qc.ipynb).


In [ ]:
dataset_name = "animal_diag_satellite_relay_tagging_location_qc_delayed_qc"

## Install/Update packages and Load common functions

In [ ]:
import os, requests, importlib.util

open('setup.py', 'w').write(requests.get('https://raw.githubusercontent.com/aodn/aodn_cloud_optimised/main/notebooks/setup.py').text)

spec = importlib.util.spec_from_file_location("setup", "setup.py")
setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(setup)

setup.install_requirements()
setup.load_dataquery()

In [ ]:
from DataQuery import GetAodn

# Understanding the Dataset

## Understanding Parquet Partitioning

Parquet files can be **partitioned** by one or more columns, which means the data is physically organised into folders based on the values in those columns. This is similar to how databases use indexes to optimise query performance.

Partitioning enables **faster filtering**: when you query data using a partitioned column, only the relevant subset of files needs to be read—improving performance significantly.

For example, if a dataset is partitioned by `"site_code"`, `"timestamp"`, and `"polygon"`, filtering on `"site_code"` allows the system to skip unrelated files entirely.

In this notebook, the `GetAodn` class includes built-in methods to efficiently filter data by **time** and **latitude/longitude** using the **timestamp** and **polygon** partitions. Other partitions can be used for filtering via the `scalar_filter`.

Any filtering on columns that are **not** partitioned can be significantly slower, as all files may need to be scanned. However, the `GetAodn` class provides a `scalar_filter` method that lets you apply these filters at load time—before the data is fully read—helping reduce the size of the resulting DataFrame.

Once the dataset is loaded, further filtering using Pandas is efficient and flexible.

See further below in the notebook for examples of how to filter the data effectively.

To view the actual partition columns for this dataset, run:


In [ ]:
aodn = GetAodn()
dname = f'{dataset_name}.parquet'
%time aodn_dataset = aodn.get_dataset(dname)

In [ ]:
aodn_dataset.dataset.partitioning.schema

## List unique partition values

In [ ]:
%%time
unique_partition_value = aodn_dataset.get_unique_partition_values('timestamp')
print(list(unique_partition_value)[0:2])  # showing a subset only

## Visualise Spatial Extent of the dataset
This section plots the polygons representing the areas where data is available. It helps to identify and create a bounding box around the regions containing data.

In [ ]:
aodn_dataset.plot_spatial_extent()

## Get Temporal Extent of the dataset

Similarly to the spatial extent, we're retrieving the minimum and maximum timestamp partition values of the dataset. This is not necessarily an accurate representation of the TIME values, as the timestamp partition can be yearly, monthly, ..., but it gives an indication of the temporal range.

In [ ]:
%%time
aodn_dataset.get_temporal_extent()

## Read Metadata

For all Parquet datasets, we create a sidecar file named **_common_metadata** in the root of the dataset. This file contains both the dataset-level and variable-level attributes.  
The metadata can be retrieved below as a dictionary, and it will also be included in the pandas DataFrame when using the `get_data` method from the `GetAodn` class.

In [ ]:
metadata = aodn_dataset.get_metadata()
metadata

# Data Query and Plot

## Create a TIME and BoundingBox filter

This cell loads a subset of the dataset based on a time range and a spatial bounding box. The result is returned as a pandas DataFrame, and basic information about its structure is displayed.

In [ ]:
%%time
df = aodn_dataset.get_data(date_start='2019-12-01',
                           date_end='2020-10-15',
                           lat_min=-60, lat_max=-50, lon_min=100, lon_max=120)

df.info()

In [ ]:
df

In [ ]:
## Download Subsetted Data as CSV

# This cell downloads the filtered dataset as a ZIP-compressed CSV file.  
# The CSV includes metadata at the top as commented lines, and a `FileLink` object is returned to allow downloading directly from the notebook.


#df.aodn.download_as_csv()

In [ ]:
import matplotlib.pyplot as plt

# Filter for a single seal to see a clean track
seal_df = df[df["ptt"] == 265888].sort_values(by="d_date")

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(
    seal_df["lon"],
    seal_df["lat"],
    c=seal_df["lq"],
    cmap="viridis",
    s=25,
    alpha=0.8,
)

# Connect the dots chronologically
ax.plot(seal_df["lon"], seal_df["lat"], color="gray", linestyle="-", alpha=0.5)

ax.set_title("Argos Track colored by Location Quality (lq)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Location Quality Index (lq)")
plt.show()

# Dynamic Argos Telemetry & Tracking of Seal PTT 265888

In [ ]:
import matplotlib.animation as animation
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import pandas as pd
from IPython.display import HTML

# =========================================================================
# 1. PURE NUMERIC DATA PREPARATION
# =========================================================================
target_ptt = 265888
seal_df = df[df["ptt"] == target_ptt].sort_values(by="d_date").copy()

# Ensure datetimes are clean and remove rows without spatial coordinates
seal_df["d_date"] = pd.to_datetime(seal_df["d_date"])
seal_df = seal_df.dropna(subset=["d_date", "lon", "lat"])
seal_df["date_only"] = seal_df["d_date"].dt.date

unique_days = sorted(seal_df["date_only"].unique())
day_groups = seal_df.groupby("date_only")

# Pre-allocate dictionary histories for line tracks AND single-point ellipse metadata
track_history = {}
ellipse_history = {}
accumulated_lon = []
accumulated_lat = []

# Scaling factor to make meters-based error attributes visible on a degree grid
scale = 0.001 

for day in unique_days:
    if day in day_groups.groups:
        day_df = day_groups.get_group(day)
        accumulated_lon.extend(day_df["lon"].tolist())
        accumulated_lat.extend(day_df["lat"].tolist())
        
        # Grab the very last known telemetry point of the day to position the active ellipse
        last_row = day_df.iloc[-1]
        current_ellipse_data = {
            "visible": True,
            "xy": (last_row["lon"], last_row["lat"]),
            "width": last_row["semi_major_axis"] * scale,
            "height": last_row["semi_minor_axis"] * scale,
            "angle": last_row["ellipse_orientation"]
        }
    else:
        # If no new data on this day, keep previous position metadata
        current_ellipse_data = current_ellipse_data.copy() if 'current_ellipse_data' in locals() else {"visible": False}

    track_history[day] = (list(accumulated_lon), list(accumulated_lat))
    ellipse_history[day] = current_ellipse_data

# =========================================================================
# 2. VANILLA CANVAS SETUP
# =========================================================================
fig, ax = plt.subplots(figsize=(12, 9))

# Set fixed bounds manually based on your data coordinates
pad = 2.0
lon_min, lon_max = seal_df["lon"].min() - pad, seal_df["lon"].max() + pad
lat_min, lat_max = seal_df["lat"].min() - pad, seal_df["lat"].max() + pad

ax.set_xlim(lon_min, lon_max)
ax.set_ylim(lat_min, lat_max)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_facecolor("aliceblue")  # Light blue background to substitute ocean

# Initialize standard tracking line
(track_line,) = ax.plot([], [], linestyle="-", linewidth=2.0, color="royalblue", alpha=0.8)

# Initialize a single error ellipse placeholder patch
active_ellipse = Ellipse(
    xy=(0, 0), width=0, height=0, angle=0,
    edgecolor="crimson", facecolor="none", linewidth=2.0, alpha=0.8
)
ax.add_patch(active_ellipse)

title_text = ax.text(0.5, 1.02, "", transform=ax.transAxes, ha="center", va="bottom", fontsize=14)
plt.tight_layout()

# =========================================================================
# 3. FAST ANIMATION UPDATE LOOP
# =========================================================================
def update(frame):
    current_day = unique_days[frame]
    
    # Update line coordinates
    lons, lats = track_history[current_day]
    track_line.set_data(lons, lats)
    
    # Update ellipse properties dynamically
    e_data = ellipse_history[current_day]
    if e_data.get("visible", False):
        active_ellipse.set_visible(True)
        active_ellipse.set_center(e_data["xy"])
        active_ellipse.set_width(e_data["width"])
        active_ellipse.set_height(e_data["height"])
        active_ellipse.set_angle(e_data["angle"])
    else:
        active_ellipse.set_visible(False)

    title_text.set_text(
        f"Seal {target_ptt} Telemetry Tracking\nDate: {current_day.strftime('%Y-%m-%d')} | Crimson Ring = Current Error Boundary"
    )
    
    return [track_line, active_ellipse, title_text]

# Run animation with blit=True for near-instant rendering
ani = animation.FuncAnimation(
    fig, update, frames=len(unique_days), interval=100, blit=True
)
plt.close()

HTML(ani.to_jshtml())